In [1]:
import os
import pyodbc
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

In [2]:
load_dotenv()

server = os.getenv("server")
database = os.getenv("database")
username = os.getenv("username")
password = os.getenv("password")
api_key = os.getenv("api_key")

### DB connection

In [3]:
server = server
database = database
username = username
password = password

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"MARS_Connection=Yes;"

)

cursor = conn.cursor()
cursor.execute("SELECT 1")
print("Connected successfully!")

Connected successfully!


### Creating tables

In [4]:
cursor.execute("""
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'dim_currencies')
    CREATE TABLE dim_currencies(
        currency VARCHAR(5) PRIMARY KEY
        , locale VARCHAR(20) NOT NULL
        , two_letter_code VARCHAR(2) NOT NULL
        , currency_name VARCHAR(20) NOT NULL
        , currency_name_short VARCHAR(20) NOT NULL
    );
""")

In [5]:
cursor.execute("""
    IF NOT EXISTS(SELECT * FROM sys.tables WHERE name = 'dim_date')
    CREATE TABLE dim_date(
        date VARCHAR(10) PRIMARY KEY,
        year INT NOT NULL,
        fiscal_year INT NOT NULL,
        month INT NOT NULL,
        day INT NOT NULL,
        quarter INT NOT NULL
    );
""")

In [6]:
cursor.execute("""
    IF NOT EXISTS(SELECT * FROM sys.tables WHERE name = 'fact_fx_rates')
    CREATE TABLE fact_fx_rates(
        date VARCHAR(10) NOT NULL
        , base_currency VARCHAR(3) NOT NULL
        , target_currency VARCHAR(3) NOT NULL
        , rate FLOAT NOT NULL
        , PRIMARY KEY(date, base_currency, target_currency)
    );
""")

In [7]:
cursor.execute("""
    IF NOT EXISTS(SELECT * FROM sys.tables WHERE name = 'raw_fx_rates')
    CREATE TABLE raw_fx_rates(
        date VARCHAR(10) PRIMARY KEY
        , EUR FLOAT NOT NULL
        , NOK FLOAT NOT NULL
        , SEK FLOAT NOT NULL
        , PLN FLOAT NOT NULL
        , RON FLOAT NOT NULL
        , DKK FLOAT NOT NULL
        , CZK FLOAT NOT NULL
    );
""")

In [8]:
conn.commit()

### Load dimensional tables 

In [4]:
import requests
import logging
import pandas as pd
from datetime import datetime, timedelta

In [10]:
base_url = "https://v6.exchangerate-api.com/v6"
currencies = ["NOK", "EUR", "SEK", "PLN", "RON", "DKK", "CZK"]

In [11]:

def load_dim_date(conn, start_date=datetime(2025, 1, 1)):
    cursor = conn.cursor()
    current = start_date
    end = datetime.today()

    while current <= end:
        fiscal_year = current.year + 1 if current.month >= 10 else current.year

        cursor.execute("""
            MERGE dim_date AS target
            USING (SELECT ? AS date) AS source
            ON target.date = source.date
            WHEN NOT MATCHED THEN
                INSERT (date, year, fiscal_year, month, day, quarter)
                VALUES (?, ?, ?, ?, ?, ?);
        """,
            current.strftime("%Y-%m-%d"),
            current.strftime("%Y-%m-%d"),
            current.year,
            fiscal_year,
            current.month,
            current.day,
            (current.month - 1) // 3 + 1
        )

        current += timedelta(days=1)

    conn.commit()
    logging.info("dim_date loaded.")

In [14]:


def load_dim_curr(conn):
    cursor = conn.cursor()

    for currency in currencies:
        response = requests.get(f"{base_url}/{api_key}/enriched/EUR/{currency}")
        response.raise_for_status()

        data = response.json()

        cursor.execute("""
            MERGE dim_currencies AS target
            USING (SELECT ? AS currency) AS source
            ON target.currency = source.currency
            WHEN NOT MATCHED THEN
                INSERT (currency, locale, two_letter_code, currency_name, currency_name_short)
                VALUES (?, ?, ?, ?, ?);
        """,
            data["target_code"],
            data["target_code"],
            data["target_data"]["locale"],
            data["target_data"]["two_letter_code"],
            data["target_data"]["currency_name"],
            data["target_data"]["currency_name_short"],
        )

    conn.commit()
    logging.info("dim_currency loaded.")



In [15]:

if __name__ == "__main__":
    load_dim_curr(conn)
    load_dim_date(conn)
    conn.close()
    print("Dimensions loaded successfully!")

Dimensions loaded successfully!


### Extract

In [5]:
import pandas as pd

In [10]:

def latest_fx(base_url, currencies, base_currency="EUR") -> pd.DataFrame:
    """
    Get daily data
    """

    base_url = "https://v6.exchangerate-api.com/v6"
    currencies = ["EUR", "NOK", "SEK", "PLN", "RON", "DKK", "CZK"]

    # Get data
    url = f"{base_url}/{api_key}/latest/{base_currency}"
    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # Calculate cross pairs
    records = []

    try:
        # Extract currencies rates
        rates = data["conversion_rates"]
        record = {"date": datetime.today().strftime("%Y-%m-%d")}

        for currency in currencies:
            record[currency] = rates.get(currency)

        records.append(record)

        # Validation
        assert (
            len(pd.DataFrame(records)) == 1
        ), f"1 row is expected, got {len(pd.DataFrame(records))}"
        return pd.DataFrame(records)

    except Exception as e:
        logging.error(f"{datetime.today().strftime('%Y-%m-%d')} error: {e}")
        return pd.DataFrame()


def history_fx(
    base_url, currencies, start_date=datetime(2026, 1, 1), base_currency="EUR"
) -> pd.DataFrame:
    """
    Get history data from start_date to end
    """

    records = []
    end_date = datetime.today()
    current_date = start_date

    while current_date < end_date:
        year = current_date.year
        month = current_date.month
        day = current_date.day

        url = f"{base_url}/{api_key}/history/{base_currency}/{year}/{month}/{day}"
        response = requests.get(url)
        data = response.json()

        try:
            rates = data["conversion_rates"]
            record = {"date": current_date.strftime("%Y-%m-%d")}

            for currency in currencies:
                record[currency] = rates.get(currency)

            records.append(record)

            current_date += timedelta(days=1)
        except Exception as e:
            logging.error(f"{datetime.today().strftime('%Y-%m-%d')} error: {e}")

    return pd.DataFrame(records)

In [7]:
def extract_run_azure():
    base_url = "https://v6.exchangerate-api.com/v6"
    currencies = ["EUR", "NOK", "SEK", "PLN", "RON", "DKK", "CZK"]

    # Check if raw_fx_rates has data
    cursor.execute("SELECT COUNT(*) FROM raw_fx_rates")
    row_count = cursor.fetchone()[0]

    if row_count == 0:
        df = history_fx(base_url, currencies)

        for _, row in df.iterrows():
            cursor.execute("""
                MERGE raw_fx_rates AS target
                USING (SELECT ? AS date) AS source
                ON target.date = source.date
                WHEN NOT MATCHED THEN
                    INSERT (date, EUR, NOK, SEK, PLN, RON, DKK, CZK)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?);
            """,
                row["date"],
                row["date"],
                row["EUR"], row["NOK"], row["SEK"],
                row["PLN"], row["RON"], row["DKK"], row["CZK"],
            )

        conn.commit()
        logging.info(f"Backfill: Extracted {len(df)} rows!")

    else:
        df = latest_fx(base_url, currencies)

        for _, row in df.iterrows():
            cursor.execute("""
                MERGE raw_fx_rates AS target
                USING (SELECT ? AS date) AS source
                ON target.date = source.date
                WHEN NOT MATCHED THEN
                    INSERT (date, EUR, NOK, SEK, PLN, RON, DKK, CZK)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?);
            """,
                row["date"],
                row["date"],
                row["EUR"], row["NOK"], row["SEK"],
                row["PLN"], row["RON"], row["DKK"], row["CZK"],
            )

        conn.commit()
        logging.info(f"Daily: Extracted {len(df)} rows!")

    conn.close()

In [11]:
extract_run_azure()

### Transform

In [6]:
import logging
import os
from datetime import datetime
from itertools import permutations

In [11]:
def transform_azure():
    cursor = conn.cursor()

    currencies = ["EUR", "NOK", "SEK", "PLN", "RON", "DKK", "CZK"]

    # Check if fact_fx_rates has data
    cursor.execute("SELECT COUNT(*) FROM fact_fx_rates")
    row_count = cursor.fetchone()[0]

    # IF EMPTY: TRANSFORM ALL RAW DATA
    if row_count == 0:
        df = pd.read_sql("SELECT * FROM raw_fx_rates", conn)

        cross_pairs = []
        try:
            for _, row in df.iterrows():
                for base, target in permutations(currencies, 2):
                    rate = row[target] / row[base]
                    cross_pairs.append({
                        "date": row["date"],
                        "base_currency": base,
                        "target_currency": target,
                        "rate": round(rate, 6),
                    })
        except Exception as e:
            logging.error(f"{datetime.today().strftime('%Y-%m-%d')} error: {e}")

        df_cross = pd.DataFrame(cross_pairs)

        # Validation
        assert df_cross["rate"].iloc[-1] > 0, \
            f"Rate should always be positive, got {df_cross['rate'].iloc[-1]}"

        for _, row in df_cross.iterrows():
            cursor.execute("""
                MERGE fact_fx_rates AS target
                USING (SELECT ? AS date, ? AS base_currency, ? AS target_currency) AS source
                ON target.date = source.date
                    AND target.base_currency = source.base_currency
                    AND target.target_currency = source.target_currency
                WHEN NOT MATCHED THEN
                    INSERT (date, base_currency, target_currency, rate)
                    VALUES (?, ?, ?, ?);
            """,
                row["date"], row["base_currency"], row["target_currency"],
                row["date"], row["base_currency"], row["target_currency"], row["rate"],
            )

        conn.commit()
        logging.info(f"Backfill: Transformed and loaded {len(df_cross)} rows!")

    # ELSE: TRANSFORM ONLY LATEST DAY
    else:
        df = pd.read_sql("""
            SELECT TOP 1 *
            FROM raw_fx_rates
            ORDER BY date DESC
        """, conn)

        cross_pairs = []
        try:
            for _, row in df.iterrows():
                for base, target in permutations(currencies, 2):
                    rate = row[target] / row[base]
                    cross_pairs.append({
                        "date": row["date"],
                        "base_currency": base,
                        "target_currency": target,
                        "rate": round(rate, 6),
                    })
        except Exception as e:
            logging.error(f"{datetime.today().strftime('%Y-%m-%d')} error: {e}")

        assert len(cross_pairs) == 42, \
            f"42 rows expected, got {len(cross_pairs)}"

        df_cross = pd.DataFrame(cross_pairs)

        assert df_cross["rate"].iloc[-1] > 0, \
            f"Rate should always be positive, got {df_cross['rate'].iloc[-1]}"

        for _, row in df_cross.iterrows():
            cursor.execute("""
                MERGE fact_fx_rates AS target
                USING (SELECT ? AS date, ? AS base_currency, ? AS target_currency) AS source
                ON target.date = source.date
                    AND target.base_currency = source.base_currency
                    AND target.target_currency = source.target_currency
                WHEN NOT MATCHED THEN
                    INSERT (date, base_currency, target_currency, rate)
                    VALUES (?, ?, ?, ?);
            """,
                row["date"], row["base_currency"], row["target_currency"],
                row["date"], row["base_currency"], row["target_currency"], row["rate"],
            )

        conn.commit()
        logging.info(f"Daily: Transformed and loaded {len(cross_pairs)} rows!")

        # Append new date to dim_date
        load_dim_date(conn, datetime.today())
        logging.info(f"Appended date {datetime.today().strftime('%Y-%m-%d')}")

    conn.close()

In [12]:
transform_azure()

/tmp/ipykernel_1305952/819253483.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM raw_fx_rates", conn)
